# Executing Circuit

This page runs a circuit on the optical quantum computer of Claveles. It is the machine-side counterpart of
the [Quickstart](quickstart.ipynb), which runs the same kind of circuit on the simulator: the circuit is
built the same way as there, and the client differs.

```{note}
The cells below submit jobs to the machine, so they need a valid `url` and `api_key` and are not run
when this documentation is built. They are shown for reference, without outputs.
```

## Executing circuit representation

First, we create the circuit below.

```{image} _images/circuit_repr_sample_circuit_light.svg
:alt: circuit
:class: only-light
```
```{image} _images/circuit_repr_sample_circuit_dark.svg
:alt: circuit
:class: only-dark
```

No state is placed on either mode, which leaves both **open**: the machine supplies its own squeezed
states, so that is the usual way to write a circuit for it. See {ref}`sec:circuit-placing-states`.

In [ ]:
from math import pi

from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import ControlledZ, Displacement, MeasureHomodyne, PhaseRotation

circuit = CircuitRepr("circuit")
circuit.Q(0) | PhaseRotation(phi=pi / 4) | Displacement(1 - 1j)
circuit.Q(0, 1) | ControlledZ(g=1)
circuit.Q(0) | MeasureHomodyne(phi=pi / 2)
circuit.Q(1) | MeasureHomodyne(phi=0)

circuit

### Configuring the client

A client says where a program runs, and carries the connection details and the execution parameters.
{py:class}`~claveles.client.OptQCClient` is the one that reaches the machine: give it the connection point,
your API key, and the `target` that matches your connection (`"cloud"` or `"direct"`; `"direct"` also
needs `qc_node_id`). [OptQC Client](optqc.md) documents every parameter, and {ref}`sec:clients` compares
this client with the simulator's.

In [ ]:
from claveles.client import OptQCClient

client = OptQCClient(
    url="https://compute.cloud.optqc.com",
    api_key="YOUR_API_KEY",
    target="cloud",
)

{py:attr}`~claveles.client.abstract.AbstractClient.n_shots` says how many times the circuit is executed. It
can be given to the constructor as well, and changed on the client afterwards:

In [ ]:
# Run the circuit for hundred times
client.n_shots = 100

### Executing the circuit

{py:func}`~claveles.execute.execute` takes a representation and a client, and returns the execution result.

In [ ]:
from claveles.execute import execute

circuit_result = execute(circuit, client)

In [ ]:
type(circuit_result)

{py:class}`~claveles.execute.ExecutionResult` has the following attributes.

| Attribute | Type | Description |
| --------- | ---- | ----------- |
| {py:attr}`~claveles.execute.ExecutionResult.total_time`         | `timedelta` | The total time to execute a representation. Measured around the call, so it is always available. |
| {py:attr}`~claveles.execute.ExecutionResult.execution_time`     | `timedelta` or None | The time the backend itself reported for executing a representation. `None` where the backend reports none, which is the case for {py:class}`~claveles.client.OptQCClient` today. |
| {py:attr}`~claveles.execute.ExecutionResult.input_repr`         | {py:class}`~claveles.circuit.CircuitRepr`, {py:class}`~claveles.graph.GraphRepr` or {py:class}`~claveles.machinery.MachineryRepr` | Executed representation.  |
| {py:attr}`~claveles.execute.ExecutionResult.execution_result`   | {py:class}`~claveles.circuit.CircuitResult`, {py:class}`~claveles.graph.GraphResult` or {py:class}`~claveles.machinery.MachineryResult` |Execution result. |
| {py:attr}`~claveles.execute.ExecutionResult.client_result`      | {py:class}`~claveles.client.abstract.AbstractClientResult` | Inherent result property for each client. |
| {py:attr}`~claveles.execute.ExecutionResult.n_shots`            | `int` | How many shots the result actually holds, read off the client result. |
| {py:attr}`~claveles.execute.ExecutionResult.requested_n_shots` | `int` | How many shots were asked for, read off the client at the moment of submission. The two agree on an ordinary run, and differ if the backend returns fewer. |

In [ ]:
from pprint import pprint

pprint(circuit_result)

When you use an {py:class}`~claveles.client.OptQCClient`, {py:attr}`~claveles.execute.ExecutionResult.client_result` is an {py:class}`~claveles.client.OptQCClientResult`.

In [ ]:
type(circuit_result.client_result)

{py:class}`~claveles.client.OptQCClientResult` has the following attributes.

The machine measures macronodes, so a machinery result is what it reports, and the other two levels are
decoded back from it. Decoding a level needs the representation it was compiled from, which is what
decides how many of the first three are populated: submitting a circuit fills all three, submitting a
graph fills the lower two, and submitting a machinery representation fills only its own.

| Attribute | Type | Description |
| --------- | ---- | ----------- |
| {py:attr}`~claveles.client.OptQCClientResult.circuit_result`    | {py:class}`~claveles.circuit.CircuitResult` or None | Measurement results in circuit representation. Populated when a circuit representation is submitted. |
| {py:attr}`~claveles.client.OptQCClientResult.graph_result`      | {py:class}`~claveles.graph.GraphResult` or None | Measurement results in graph representation. Populated when a circuit or a graph representation is submitted. |
| {py:attr}`~claveles.client.OptQCClientResult.machinery_result`  | {py:class}`~claveles.machinery.MachineryResult` or None | Measurement results in machinery representation. Populated for every job, whichever representation was submitted: this is the level the machine reports, and the one the other two are decoded from. |
| {py:attr}`~claveles.client.OptQCClientResult.execution_result`  | {py:class}`~claveles.circuit.CircuitResult`, {py:class}`~claveles.graph.GraphResult` or {py:class}`~claveles.machinery.MachineryResult` | The highest-level available result (priority: circuit > graph > machinery), which is the level that was submitted. |
| {py:attr}`~claveles.client.OptQCClientResult.execution_time`    | None | Always `None`: the job endpoints this client reads carry a status and a result location, and no timing. Use {py:attr}`~claveles.execute.ExecutionResult.total_time` for the duration of the call. |
| {py:attr}`~claveles.client.OptQCClientResult.details`           | `dict` | Additional execution details, such as timing information. Empty today, for the same reason {py:attr}`~claveles.client.OptQCClientResult.execution_time` is `None`. |

When you execute the circuit representation, you can get the result of the circuit representation for any shot by using index access.

In [ ]:
pprint(circuit_result[0])  # Get the 0-th shot result

In [ ]:
pprint(circuit_result[1])  # Get the 1-st shot result

Each {py:class}`~claveles.circuit.result.CircuitModeMeasuredValue` carries the mode it read as {py:attr}`~claveles.circuit.result.CircuitModeMeasuredValue.mode` and the value as {py:attr}`~claveles.circuit.result.CircuitModeMeasuredValue.value`. A measurement consumes the mode it reads, so a mode appears at most once per shot.

(sec:execution-inspecting-the-compilation)=

## Inspecting what the circuit is compiled into

A circuit representation is not what the machine runs. The client compiles it, first to a graph
representation and then to a machinery representation, and the result carries all three levels. You can
reproduce those two steps locally with {py:class}`~claveles.client.OptQCCompiler`, which performs exactly
the conversions the client performs, to see what will be executed before submitting anything:

In [ ]:
from claveles.client import OptQCCompiler

compiler = OptQCCompiler()
graph = compiler.circuit_to_graph(circuit)

{py:func}`~claveles.graph.visualize.make_figure` draws the graph. `ignore_wiring=True` hides the rows and
columns that carry nothing but wiring, of which a compiled graph has many; see
[Visualizing Graph Representation](viz_graph_repr.ipynb).

In [ ]:
from claveles.graph.visualize import make_figure

make_figure(graph, ignore_wiring=True);

{py:meth}`~claveles.client.OptQCCompiler.graph_to_machinery` takes the second step, to the measurement
angles the machine is driven with:

In [ ]:
machinery = compiler.graph_to_machinery(graph)

pprint(machinery.get_homodyne_angle(0))  # Measurement angles at macronode index 0.
pprint(machinery.get_homodyne_angle(1))  # Measurement angles at macronode index 1.

The result of a run reports at all three levels too. Alongside
{py:attr}`~claveles.client.OptQCClientResult.circuit_result`, which is what
`circuit_result[0]` above reads, the graph and machinery results are available in
{py:attr}`~claveles.client.OptQCClientResult.graph_result` and
{py:attr}`~claveles.client.OptQCClientResult.machinery_result`. Those two report per **macronode** rather
than per mode, because that is what the machine measures:

In [ ]:
from claveles.client import OptQCClientResult

assert isinstance(circuit_result.client_result, OptQCClientResult)

graph_result = circuit_result.client_result.graph_result
assert graph_result is not None
print(graph_result[0])  # Macronode measured values of the first shot.

machinery_result = circuit_result.client_result.machinery_result
assert machinery_result is not None
print(machinery_result[0])  # Macronode measured values of the first shot.

```{seealso}
{ref}`sec:simulation-graph-machinery-results` for how the three result types are read, and
[Constructing Graph Representation](graph_repr.ipynb) and
[Constructing Machinery Representation](machinery_repr.ipynb) for writing either representation
yourself instead of having it compiled.
```

(sec:execution-resource-and-initialized-states)=

## Resource and initialized states

The machine only utilizes squeezed states as resources, which it entangles into the cluster state it
computes on. These are used for the mode initialization, as explained in {ref}`the graph.ops.Initialization entry <sec:gate-graph-repr-initialization>`,
and the initialized modes are also squeezed states.

This is what a mode left **open** is asking for: the circuit places no state on it, and the machine
supplies one of these. Leaving modes open is therefore the usual way to write a circuit for hardware
execution — see {ref}`sec:circuit-placing-states`. The one state worth placing explicitly is
{py:meth}`~claveles.circuit.state.QuantumState.hardware_squeezed`, which sets the squeezing angle; any other
state is refused on conversion.

The table below gives the parameters of the resource modes and of the initialized modes, per
representation. The figures are the machine's design values rather than measured ones.

| Representation | Parameter | Resource | Initialized |
| --- | --- | --- | --- |
| Circuit | Squeezing level | (no concept of a resource) | $(5-10\log_{10} 2)$ dB (approx.) |
| Circuit | Anti-squeezing level | (no concept of a resource) | $(10-10\log_{10} 2)$ dB (approx.) |
| Circuit | Squeezing angle | (no concept of a resource) | the `phi` parameter of {py:meth}`~claveles.circuit.state.QuantumState.hardware_squeezed` |
| Graph | Squeezing level | 5 dB | $(5-10\log_{10} 2)$ dB (approx.) |
| Graph | Anti-squeezing level | 10 dB | $(10-10\log_{10} 2)$ dB (approx.) |
| Graph | Squeezing angle | 0 | $\theta+\pi/2$, where $\theta$ is the parameter of {py:class}`~claveles.graph.ops.Initialization` |
| Machinery | Squeezing level | 5 dB | (not emphasized; see below) |
| Machinery | Anti-squeezing level | 10 dB | (not emphasized; see below) |
| Machinery | Squeezing angle | 0 | (not emphasized; see below) |

```{seealso}
See {ref}`sec:squeezing-level-and-squeezing-angle` for the definitions of squeezing level and squeezing
angle.
```

A resource mode is **not** a pure state, as the differing squeezing and anti-squeezing levels in the
table show. This occurs due to optical loss and noise arising from finite squeezing.

In the **machinery** representation, mode initialization is just one of the operations that can be
realized through homodyne angle configuration, as described in
{ref}`the graph.ops.Initialization entry <sec:gate-graph-repr-initialization>`, so the state after initialization is not treated as
something separate — which is why the "Initialized" column says nothing for it.

In the **graph** representation, the resource mode is the state prior to initialization. Once
initialized, the mode becomes manageable through the operations defined within the graph
representation, and the "Initialized" column gives the parameters of the mode after initialization. Of
those, only the squeezing angle is under your control, through the `theta` parameter of
{py:class}`~claveles.graph.ops.Initialization`.

In the **circuit** representation, only the modes after initialization are visible, and the squeezing
angle is specified through the parameter of
{py:meth}`~claveles.circuit.state.QuantumState.hardware_squeezed`.